# 00 - Kenalan dengan Data (untuk SEMUA anggota)

**Tujuan:** memastikan Colab kamu bisa membuka repo, membaca data, dan kamu paham isi tiap kolom.
Jalankan sel dari atas ke bawah dengan **Shift + Enter**. Kalau semua sel hijau tanpa error, kamu siap kerja.

> Tema B - Kualitas Udara & Mobilitas Perkotaan | Kelompok 12

In [1]:
# ============ SETUP (jalankan sel ini PERTAMA) ============
# Satu-satunya yang boleh diubah: GITHUB_USER dan GITHUB_REPO (diisi sekali oleh Project Lead).
import os, sys, subprocess
GITHUB_USER = "Tur-ai"      # <- username GitHub pemilik repo
GITHUB_REPO = "udara-mobilitas-statsprob"     # <- nama repository
BRANCH = "main"

if "google.colab" in sys.modules:                      # kalau dijalankan di Google Colab
    PROYEK = "/content/proyek"
    if not os.path.exists(PROYEK):
        token = ""
        try:                                           # opsional: untuk repo PRIVATE (lihat docs/02_...)
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:
            pass
        url = f"https://{token}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, PROYEK],
                           capture_output=True, text=True)
        if r.returncode != 0:
            pesan = (r.stderr or "").replace(token.rstrip("@"), "***") if token else (r.stderr or "")
            raise RuntimeError("Gagal clone repo. Cek GITHUB_USER/GITHUB_REPO, atau lihat bagian "
                               "'Troubleshooting' di docs/02_panduan_github_colab.md\n" + pesan)
    else:
        subprocess.run(["git", "-C", PROYEK, "pull"], capture_output=True)   # ambil update terbaru
    os.chdir(PROYEK)
elif os.path.basename(os.getcwd()) == "notebooks":    # kalau dijalankan di Jupyter lokal
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.helpers import *
sns.set_theme(style="whitegrid", font_scale=1.0)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Setup selesai. Folder kerja:", os.getcwd())

Setup selesai. Folder kerja: d:\Kuliah\Semester 3\StatsProb\udara-mobilitas-statsprob


## 1. Lihat 5 baris pertama
Setiap baris = 1 observasi (1 pengukuran di 1 lokasi pada 1 waktu).

In [2]:
df = muat_data_mentah()
df.head()

,ID_Observasi,Zona_Lokasi,Jenis_Hari,Jam_Sibuk,Volume_Kendaraan,Kecepatan_Angin_KmJ,Suhu_Celcius,Indeks_PM25,Status_Udara
0,OBS-12-0001,Pusat Kota,Hari Kerja,Tidak,623,35.0,27.7,35.8,Sedang
1,OBS-12-0002,Kawasan Kampus,Hari Kerja,Tidak,433,11.8,30.3,45.8,Sedang
2,OBS-12-0003,Pusat Kota,Hari Kerja,Tidak,670,16.2,25.2,54.3,Sedang
3,OBS-12-0004,Pusat Kota,Akhir Pekan,Tidak,487,3.8,27.9,51.5,Sedang
4,OBS-12-0005,Pinggiran,Hari Kerja,Tidak,273,6.6,29.6,28.8,Baik


## 2. Ukuran data dan tipe kolom

In [3]:
print("Baris x Kolom:", df.shape)
df.info()

Baris x Kolom: (350, 9)
<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_Observasi         350 non-null    str    
 1   Zona_Lokasi          350 non-null    str    
 2   Jenis_Hari           350 non-null    str    
 3   Jam_Sibuk            350 non-null    str    
 4   Volume_Kendaraan     350 non-null    int64  
 5   Kecepatan_Angin_KmJ  347 non-null    float64
 6   Suhu_Celcius         350 non-null    float64
 7   Indeks_PM25          347 non-null    float64
 8   Status_Udara         350 non-null    str    
dtypes: float64(3), int64(1), str(5)
memory usage: 24.7 KB


## 3. Arti tiap kolom (kamus data)

| Kolom | Arti | Jenis |
|---|---|---|
| `ID_Observasi` | nomor pengenal baris | pengenal (tidak dianalisis) |
| `Zona_Lokasi` | Pusat Kota / Kawasan Kampus / Pinggiran | kategori |
| `Jenis_Hari` | Hari Kerja / Akhir Pekan | kategori (2 kelompok) |
| `Jam_Sibuk` | Ya / Tidak | kategori (2 kelompok) |
| `Volume_Kendaraan` | jumlah kendaraan yang lewat | angka |
| `Kecepatan_Angin_KmJ` | kecepatan angin (km/jam) | angka |
| `Suhu_Celcius` | suhu udara (derajat C) | angka |
| `Indeks_PM25` | **kadar partikel halus PM2.5 - variabel utama kita** | angka |
| `Status_Udara` | Baik / Sedang / Tidak Sehat (turunan dari `Indeks_PM25`) | kategori berurutan |

**PM2.5** = partikel debu sangat halus (lebih kecil dari 2,5 mikrometer) yang bisa masuk paru-paru. Makin besar angkanya, makin buruk udaranya.

## 4. Cek cepat: ada data kosong?

In [4]:
df.isna().sum()

ID_Observasi           0
Zona_Lokasi            0
Jenis_Hari             0
Jam_Sibuk              0
Volume_Kendaraan       0
Kecepatan_Angin_KmJ    3
Suhu_Celcius           0
Indeks_PM25            3
Status_Udara           0
dtype: int64

## 5. Ringkasan angka dan hitungan kategori

In [5]:
display(df[KOLOM_NUMERIK].describe().round(2))
for k in KOLOM_KATEGORI:
    print("\n" + k)
    print(df[k].value_counts())

,Volume_Kendaraan,Kecepatan_Angin_KmJ,Suhu_Celcius,Indeks_PM25
count,350.00,347.00,350.00,347.00
mean,408.77,10.05,28.69,43.12
std,163.81,6.48,2.43,23.19
min,50.00,1.00,21.00,5.00
25%,292.00,5.45,27.10,29.90
50%,403.50,8.70,28.65,43.20
75%,532.75,13.00,30.30,54.15
max,813.00,35.00,35.60,280.50



Zona_Lokasi
Zona_Lokasi
Pusat Kota        138
Kawasan Kampus    132
Pinggiran          80
Name: count, dtype: int64

Jenis_Hari
Jenis_Hari
Hari Kerja     233
Akhir Pekan    117
Name: count, dtype: int64

Jam_Sibuk
Jam_Sibuk
Tidak    210
Ya       140
Name: count, dtype: int64

Status_Udara
Status_Udara
Sedang         195
Baik           127
Tidak Sehat     28
Name: count, dtype: int64


## 6. Tes fungsi bersama
Sel ini memuat data bersih. Kalau Data Engineer belum meng-upload data bersih resmi, kamu akan melihat peringatan **DRAF SEMENTARA** - itu normal di awal.

In [6]:
df_bersih = muat_data_bersih()
df_bersih.head()

Sumber data : !!! DRAF SEMENTARA !!! (data bersih resmi belum di-upload; jangan pakai untuk laporan final)
Jumlah baris: 345 | Jumlah kolom: 9


,ID_Observasi,Zona_Lokasi,Jenis_Hari,Jam_Sibuk,Volume_Kendaraan,Kecepatan_Angin_KmJ,Suhu_Celcius,Indeks_PM25,Status_Udara
0,OBS-12-0001,Pusat Kota,Hari Kerja,Tidak,623,35.0,27.7,35.8,Sedang
1,OBS-12-0002,Kawasan Kampus,Hari Kerja,Tidak,433,11.8,30.3,45.8,Sedang
2,OBS-12-0003,Pusat Kota,Hari Kerja,Tidak,670,16.2,25.2,54.3,Sedang
3,OBS-12-0004,Pusat Kota,Akhir Pekan,Tidak,487,3.8,27.9,51.5,Sedang
4,OBS-12-0005,Pinggiran,Hari Kerja,Tidak,273,6.6,29.6,28.8,Baik


### Selesai?
- [ ] Semua sel di atas jalan tanpa error
- [ ] Aku bisa menjelaskan arti tiap kolom ke teman
- [ ] Aku sudah membaca file panduan peranku di `docs/peran/`

Kalau error di sel SETUP, buka `docs/02_panduan_github_colab.md` bagian **Troubleshooting**.